In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")


In [ ]:
import os, sys, logging

sys.path.append("../")

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
from rich.console import Console
from rich.theme import Theme
from src.utils import env_utils, logging_utils

custom_theme = Theme({"default": "white"})
console = Console(theme=custom_theme)

def rich_print(*args, **kwargs):
    console.print(*args, **kwargs)

logging.basicConfig(
    level=logging.WARNING,
    format=logging_utils.DEFAULT_FORMAT,
    datefmt=logging_utils.DEFAULT_DATEFMT,
    stream=sys.stdout,
)


# Well-Known Task Completions — Inspection

Inspect outputs of `scripts/generate_well_known_task_completions.py`.

Each result file is a JSON dict with keys:
- `function_name` — task identifier
- `prompt` — full chat-template prompt sent to the model
- `generation` — raw LM completion


In [ ]:
import json
from pathlib import Path
from src.utils.env_utils import DEFAULT_RESULTS_DIR, DEFAULT_DATA_DIR

MODEL_SHORT = "Qwen2.5-Coder-32B-Instruct"   # change to inspect a different model

RESULTS_DIR = Path(DEFAULT_RESULTS_DIR) / "well-known-task-generations" / MODEL_SHORT
TASKS_DIR   = Path(DEFAULT_DATA_DIR) / "well-known-tasks"

print(f"Results dir : {RESULTS_DIR}")
print(f"Exists      : {RESULTS_DIR.exists()}")


In [ ]:
results: dict[str, dict] = {}

if RESULTS_DIR.exists():
    for path in sorted(RESULTS_DIR.glob("*.json")):
        with open(path) as f:
            results[path.stem] = json.load(f)

print(f"Loaded {len(results)} generated results")


In [ ]:
from rich.table import Table

# All available task names
all_tasks = sorted(p.stem for p in TASKS_DIR.glob("*.json"))

generated   = sorted(results.keys())
missing     = [t for t in all_tasks if t not in results]

table = Table(title=f"Coverage — {MODEL_SHORT}", show_lines=False, highlight=False)
table.add_column("Stat", style="bold")
table.add_column("Count", justify="right")
table.add_row("Total tasks available", str(len(all_tasks)))
table.add_row("Generated",            f"[green]{len(generated)}[/green]")
table.add_row("Missing",              f"[red]{len(missing)}[/red]")
rich_print(table)

if missing:
    rich_print(f"\n[red]Missing tasks:[/red] {', '.join(missing)}")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

gen_lengths = [len(r["generation"]) for r in results.values()]

fig, ax = plt.subplots(figsize=(10, 3))
ax.hist(gen_lengths, bins=40, color="steelblue", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Generation length (chars)")
ax.set_ylabel("# tasks")
ax.set_title(f"Generation length distribution — {MODEL_SHORT}  (n={len(gen_lengths)})")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"  min={min(gen_lengths)}  median={int(np.median(gen_lengths))}  "
      f"max={max(gen_lengths)}  mean={np.mean(gen_lengths):.0f}")


In [ ]:
# ── Inspect a single generation ──────────────────────────────────────────────
INSPECT_TASK = "bfs"   # change to any task name

if INSPECT_TASK not in results:
    rich_print(f"[red]{INSPECT_TASK!r} not found in results[/red]")
else:
    r = results[INSPECT_TASK]
    rich_print(f"[bold cyan]Task:[/bold cyan] {r['function_name']}")
    rich_print(f"\n[bold]Prompt:[/bold]")
    rich_print(f"[white]{r['prompt']}[/white]")
    rich_print(f"\n[bold]Generation:[/bold]")
    rich_print(f"[bright_yellow]{r['generation']}[/bright_yellow]")


In [ ]:
# ── Summary table: function name + first line of generation ──────────────────
from rich.table import Table

table = Table(title=f"All generations — {MODEL_SHORT}", show_lines=True, highlight=False)
table.add_column("Task", style="cyan", no_wrap=True)
table.add_column("Gen length", justify="right")
table.add_column("First line of generation", no_wrap=False, max_width=80)

for name, r in sorted(results.items()):
    first_line = r["generation"].strip().splitlines()[0] if r["generation"].strip() else "(empty)"
    table.add_row(name, str(len(r["generation"])), first_line)

rich_print(table)
